# Agents the Hard Way

In this exercise, we will build a simple AI agent from scratch. 

In [ ]:
# The Workbench image includes the workshop dependencies. Check Workshop Health if imports fail.
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".project/spec.yaml").exists())
MODULE_DIR = PROJECT_ROOT / "code/1-build-an-agent"
sys.path.insert(0, str(PROJECT_ROOT / "code"))
sys.path.insert(0, str(MODULE_DIR))
from workshop_support import get_model, load_secrets
load_secrets(PROJECT_ROOT)

## Setup the environment

Like most Python code, we start by importing the necessary modules.

The `openai` module is maintained by OpenAI and is used for talking to models running remotely.

The `call_llm` function is a helper function to make our code easier to read. It will format the inputs and outputs when calling the models.

In [ ]:
import json
import os
from openai import OpenAI

def call_llm(model_client, model_name, message_history, tool_list):
    """Simple wrapper for OpenAI API Chat Completion requests."""
    kwargs = {
        "model": model_name,
        "messages": message_history,
        "max_tokens": 4096,
    }
    
    if tool_list:
        kwargs["tools"] = tool_list
    
    response = model_client.chat.completions.create(**kwargs)
    message = response.choices[0].message
    
    result = {"role": "assistant", "content": message.content}
    
    if hasattr(message, 'tool_calls') and message.tool_calls:
        result["tool_calls"] = []
        for tool_call in message.tool_calls:
            result["tool_calls"].append({
                "id": tool_call.id,
                "function": {
                    "name": tool_call.function.name,
                    "arguments": tool_call.function.arguments
                },
                "type": "function"
            })
        result["content"] = None
    
    return result

## Load the configuration

Now I'll load our configuration as constants:
- `API_KEY` loads our credentials from an environment variable
- `MODEL_URL` points to the server hosting your model
- `MODEL_NAME` is the model we are going to use

In [ ]:
API_KEY = os.environ["NVIDIA_API_KEY"]
MODEL_URL = "https://integrate.api.nvidia.com/v1"
MODEL_NAME = get_model("chat")

## Part 1 - The Model
We will study four useful components: model, tools, memory, and routing. They are a teaching model, not a universal definition of every agent.

NVIDIA’s hosted endpoint supports the OpenAI-compatible Chat Completions protocol, so we can use the `OpenAI` client with NVIDIA’s URL and key. Other providers may require their own clients or support different features.

In [ ]:
# Using the MODEL_URL and API_KEY defined above, create a new model client named client.
client = OpenAI(
    base_url=..., 
    api_key=...
)

<details>
<summary>💡 NEED SOME HELP?</summary>

```python
client = OpenAI(
    base_url=MODEL_URL, 
    api_key=API_KEY
)
```
</details>

## Why these values?
`MODEL_URL` points to NVIDIA’s hosted API. The workshop’s shared model configuration chooses a model with tool calling. Choose models by task quality, supported features, latency, and cost; parameter count alone does not determine suitability.

## Part 2 - Tools
Tools are code the application can execute at the model’s request. They let our agent interact with the world. We will start with a small arithmetic tool.

In [ ]:
# Create a function, called `add`, that adds two integers, called `a` and `b`.
def add(a, b):
    ...

<details>
<summary>💡 NEED SOME HELP?</summary>

```python
def add(a, b):
    return a + b
```
</details>

## Describe the tools

Before moving on, we need to create a description of the tools that the model can understand.
Think of this as the LLM's menu of possible helpers.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Add two integers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer", "description": "First integer"},
                    "b": {"type": "integer", "description": "Second integer"},
                },
                "required": ["a", "b"],
            },
        },
    }
]

## Part 3 - Memory
We use a list of messages as short-term conversation state. Each model call receives that list explicitly; the API does not automatically remember previous calls. Some applications also persist state across sessions or retrieve long-term memories.

Start the conversation with `What is 3 plus 12?`.

In [ ]:
memory = [
  {"role": "user", "content": "What is 3 plus 12?"}
]

## Run the agent

We now have three of the four pieces required of an agent. The last missing piece is the routing. For the time being, we will forgo this piece and manually route this message.

The agent starts by giving the memory and tool list to the model. The model will reply by either requesting a tool or answering the question.

Based on the model's response, we will decide how to proceed.

Call the model using the `call_llm` function. The function takes four arguments:

- `model_client`: the OpenAI client
- `model_name`: the name of the model to use (check the constants from before)
- `message_history`: your short term memory
- `tool_list`: the menu of tools the model can access

Use this function to call the LLM. Save the result by appending it to the end of `memory`.

In [ ]:
llm_response = call_llm(
    model_client=..., 
    model_name=..., 
    message_history=..., 
    tool_list=...
)

memory.append(llm_response)
print(llm_response)

<details>
<summary>💡 NEED SOME HELP?</summary>

```python
llm_response = call_llm(
    model_client=client, 
    model_name=MODEL_NAME, 
    message_history=memory, 
    tool_list=tools
)

memory.append(llm_response)
print(llm_response)
```
</details>

## Part 4 - Routing
Inspect `tool_calls`: the model may answer directly, request one tool, or request several. The following cells handle all requested calls; if there are none, they do nothing.

Extract each function’s name, arguments, and call ID. `arguments` is a JSON string, so decode it with `json.loads`.

In [ ]:
tool_requests = []
for tool_call in llm_response.get("tool_calls", []):
    tool_name = ...
    tool_args = json.loads(...)
    tool_id = ...
    tool_requests.append({"name": tool_name, "args": tool_args, "id": tool_id})
print(f"Requested tools: {len(tool_requests)}")

<details>
<summary>💡 NEED SOME HELP?</summary>

```python
tool_requests = []
for tool_call in llm_response.get("tool_calls", []):
    tool_name = tool_call["function"]["name"]
    tool_args = json.loads(tool_call["function"]["arguments"])
    tool_id = tool_call["id"]
    tool_requests.append({"name": tool_name, "args": tool_args, "id": tool_id})
print(f"Requested tools: {len(tool_requests)}")
```
</details>

The model requests tools; our Python program executes them. For each request, call `add` with the decoded arguments. The provided guard also returns an error for an unknown tool instead of executing arbitrary code.

In [ ]:
tool_results = []
for request in tool_requests:
    tool_name, tool_args = request["name"], request["args"]
    try:
        if tool_name == "add":
            tool_out = ...
        else:
            tool_out = f"Unknown tool: {tool_name}"
    except (TypeError, ValueError) as exc:
        tool_out = f"Invalid tool arguments: {type(exc).__name__}"
    tool_results.append({"id": request["id"], "name": tool_name, "output": tool_out})
print(tool_results)

<details>
<summary>💡 NEED SOME HELP?</summary>

```python
tool_results = []
for request in tool_requests:
    tool_name, tool_args = request["name"], request["args"]
    try:
        if tool_name == "add":
            tool_out = add(**tool_args)
        else:
            tool_out = f"Unknown tool: {tool_name}"
    except (TypeError, ValueError) as exc:
        tool_out = f"Invalid tool arguments: {type(exc).__name__}"
    tool_results.append({"id": request["id"], "name": tool_name, "output": tool_out})
print(tool_results)
```
</details>

Append one tool response for every requested tool call, preserving the matching call ID.

In [ ]:
for result in tool_results:
    memory.append({
        "role": "tool",
        "tool_call_id": ...,
        "name": ...,
        "content": str(...),
    })

<details>
<summary>💡 NEED SOME HELP?</summary>

```python
for result in tool_results:
    memory.append({
        "role": "tool",
        "tool_call_id": result["id"],
        "name": result["name"],
        "content": str(result["output"]),
    })
```
</details>

Next time we call the model, it will see:
- the prompt from the user
- its own request for a tool call
- and the result of that tool call.

Before continuing, lets inspect the memory.

In [ ]:
import pprint
pprint.pprint(memory)

If tools ran, call the model again with the updated memory. If it answered directly, keep that answer. Complete the same four arguments as before.

If this new response requests more tools, repeat the routing cells (21–32) with the new response. We are manually stepping through the loop; the next exercise uses a framework to run it automatically.

In [ ]:
if tool_results:
    llm_response = call_llm(
        model_client=...,
        model_name=...,
        message_history=...,
        tool_list=...,
    )
    memory.append(llm_response)
print(llm_response)
if llm_response.get("tool_calls"):
    print("More tools requested: repeat the routing cells before expecting a final answer.")

## Complete Conversation History

Let's see the full memory to understand how the agent processed the conversation.

In [ ]:
pprint.pprint(memory)

## You have traced an agent iteration
You have passed conversation state to a model, inspected its decision, and returned tool results. A complete agent runner repeats this process until an answer or a configured step limit. Continue to the report agent to use a framework-managed loop.